# Track 3: Object Swaps and Loss

This notebook builds the CHAIR-compatible object co-occurrence matrix, generates structured and random-swap negatives, spot-checks 100 negatives, and smoke-tests the object-swap loss with pretrained ClipCap.

## Setup

Run the Track 1 setup first so Drive is mounted and the ClipCap model/tokenizer are available for the final loss cell.

In [ ]:
from __future__ import annotations

import json
import os
import pickle
import random
import re
import subprocess
import sys
from collections.abc import Mapping, Sequence
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from google.colab import drive
import transformers
from transformers import GPT2Tokenizer

drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/sc4001_nndl_grp_project'
TRACK1_ROOT = os.path.join(PROJECT_ROOT, 'phase_1', 'track_1')
TRACK3_ROOT = os.path.join(PROJECT_ROOT, 'phase_1', 'track_3')
os.makedirs(TRACK3_ROOT, exist_ok=True)

transformers.AdamW = torch.optim.AdamW
clipcap_dir = '/content/CLIP_prefix_caption'
if not os.path.isdir(clipcap_dir):
    subprocess.run(
        ['git', 'clone', '-q', 'https://github.com/rmokady/CLIP_prefix_caption', clipcap_dir],
        check=True,
    )
if clipcap_dir not in sys.path:
    sys.path.append(clipcap_dir)
from train import ClipCaptionPrefix, MappingType

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def load_object_data(path: str | Path) -> dict:
    """Load and validate Track 1's coco_objects.json."""
    with open(path) as handle:
        data = json.load(handle)
    required = {'categories', 'synonyms', 'word_to_node', 'double_words'}
    missing = required.difference(data)
    if missing:
        raise ValueError(f'object data is missing keys: {sorted(missing)}')
    return data


def load_chair_gt(path: str | Path) -> dict:
    """Load Track 1's chair_gt_<split>.pkl cache."""
    with open(path, 'rb') as handle:
        return pickle.load(handle)


def true_objects(cache_entry: Mapping) -> set[str]:
    """Return segmentation objects unioned with every human-caption object."""
    objects = set(cache_entry.get('seg', []))
    for caption_objects in cache_entry.get('cap', []):
        objects.update(caption_objects)
    return objects


def build_cooccurrence_matrix(
    chair_gt: Mapping[int, Mapping],
    categories: Sequence[str],
) -> tuple[np.ndarray, dict[str, int]]:
    """Count image-level object co-occurrences, including the diagonal."""
    category_to_index = {category: i for i, category in enumerate(categories)}
    matrix = np.zeros((len(categories), len(categories)), dtype=np.int64)
    for entry in chair_gt.values():
        present = sorted(
            category_to_index[category]
            for category in true_objects(entry)
            if category in category_to_index
        )
        for row in present:
            matrix[row, present] += 1
    return matrix, category_to_index


def save_cooccurrence_matrix(
    path: str | Path,
    matrix: np.ndarray,
    categories: Sequence[str],
) -> None:
    """Save a portable matrix artifact with its category ordering."""
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    with open(target, 'wb') as handle:
        pickle.dump({'categories': list(categories), 'matrix': matrix}, handle)


def _replacement_patterns(object_data: Mapping) -> list[tuple[re.Pattern, str]]:
    word_to_node = object_data['word_to_node']
    phrases = sorted(
        word_to_node,
        key=lambda phrase: (len(phrase.split()), len(phrase)),
        reverse=True,
    )
    return [
        (
            re.compile(
                r'(?<![A-Za-z])' + re.escape(phrase) + r'(?![A-Za-z])',
                re.IGNORECASE,
            ),
            word_to_node[phrase],
        )
        for phrase in phrases
    ]


def _object_mentions(caption: str, object_data: Mapping) -> list[tuple[str, str, int, int]]:
    """Find non-overlapping synonym spans as text, category, and offsets."""
    occupied = []
    mentions = []
    for pattern, category in _replacement_patterns(object_data):
        for match in pattern.finditer(caption):
            start, end = match.span()
            if any(start < occupied_end and occupied_start < end
                   for occupied_start, occupied_end in occupied):
                continue
            occupied.append((start, end))
            mentions.append((match.group(0), category, start, end))
    return sorted(mentions, key=lambda mention: mention[2])


def _candidate_categories(
    source: str,
    image_objects: set[str],
    categories: Sequence[str],
    matrix: np.ndarray | None,
    category_to_index: Mapping[str, int] | None,
    random_swap: bool,
    rng: random.Random,
) -> list[str]:
    candidates = [
        category for category in categories
        if category != source and category not in image_objects
    ]
    if random_swap or matrix is None or category_to_index is None or source not in category_to_index:
        rng.shuffle(candidates)
        return candidates
    source_row = matrix[category_to_index[source]]
    return sorted(
        candidates,
        key=lambda category: (source_row[category_to_index[category]], category),
    )


def generate_swap(
    caption: str,
    image_objects: set[str],
    object_data: Mapping,
    matrix: np.ndarray | None = None,
    category_to_index: Mapping[str, int] | None = None,
    *,
    random_swap: bool = False,
    rng: random.Random | None = None,
) -> dict | None:
    """Generate one negative by replacing an object with an absent category."""
    rng = rng or random.Random()
    mentions = _object_mentions(caption, object_data)
    if not mentions:
        return None
    source_text, source, start, end = rng.choice(mentions)
    candidates = _candidate_categories(
        source,
        image_objects,
        object_data['categories'],
        matrix,
        category_to_index,
        random_swap,
        rng,
    )
    if not candidates:
        return None
    target = candidates[0]
    return {
        'caption': caption[:start] + target + caption[end:],
        'source': source,
        'target': target,
        'source_text': source_text,
        'random_swap': random_swap,
    }


def spot_check_swaps(
    examples: Sequence[tuple[str, set[str]]],
    object_data: Mapping,
    matrix: np.ndarray,
    category_to_index: Mapping[str, int],
    *,
    n: int = 100,
    seed: int = 0,
    random_swap: bool = False,
) -> dict:
    """Generate and validate up to 100 structured or random negatives."""
    rng = random.Random(seed)
    selected = list(examples)
    rng.shuffle(selected)
    selected = selected[:n]
    checked = 0
    failures = []
    for caption, image_objects in selected:
        result = generate_swap(
            caption,
            image_objects,
            object_data,
            matrix,
            category_to_index,
            random_swap=random_swap,
            rng=rng,
        )
        if result is None:
            continue
        checked += 1
        if result['target'] in image_objects or result['target'] == result['source']:
            failures.append(result)
    return {'requested': len(selected), 'checked': checked, 'failures': failures}


def caption_nll(model, tokenizer, prefix, caption: str, prefix_length: int = 40):
    """Teacher-forced GPT-2 NLL for one caption and one raw CLIP feature."""
    if prefix.ndim == 1:
        prefix = prefix.unsqueeze(0)
    prefix = prefix.float()
    prefix = prefix / prefix.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    token_ids = tokenizer.encode(caption, add_special_tokens=False)
    if tokenizer.eos_token_id is not None:
        token_ids.append(tokenizer.eos_token_id)
    if not token_ids:
        raise ValueError('caption produced no tokens')
    tokens = torch.tensor(token_ids, device=prefix.device, dtype=torch.long).unsqueeze(0)
    prefix_embeds = model.clip_project(prefix).reshape(1, prefix_length, -1)
    token_embeds = model.gpt.transformer.wte(tokens)
    inputs_embeds = torch.cat((prefix_embeds, token_embeds), dim=1)
    labels = torch.cat(
        (
            torch.full((1, prefix_length), -100, device=tokens.device, dtype=torch.long),
            tokens,
        ),
        dim=1,
    )
    return model.gpt(inputs_embeds=inputs_embeds, labels=labels).loss


def object_swap_loss(
    model,
    tokenizer,
    prefixes,
    captions: Sequence[str],
    negative_captions: Sequence[str],
    *,
    prefix_length: int = 40,
    margin: float = 0.2,
    ranking_weight: float = 1.0,
):
    """Return positive NLL, negative NLL, ranking term, and total loss."""
    if len(captions) != len(negative_captions) or len(captions) != len(prefixes):
        raise ValueError('prefixes, captions, and negative_captions must have equal length')
    positive = torch.stack([
        caption_nll(model, tokenizer, prefixes[i], captions[i], prefix_length)
        for i in range(len(captions))
    ])
    negative = torch.stack([
        caption_nll(model, tokenizer, prefixes[i], negative_captions[i], prefix_length)
        for i in range(len(captions))
    ])
    ranking = F.relu(margin + positive - negative).mean()
    return {
        'positive_nll': positive.mean(),
        'negative_nll': negative.mean(),
        'ranking_loss': ranking,
        'loss': positive.mean() + ranking_weight * ranking,
    }


def smoke_test_loss(model, tokenizer, prefixes) -> dict[str, float]:
    """Run the loss on two captions using a loaded pretrained ClipCap model."""
    captions = ['a dog beside a person', 'a bird on a tree']
    negatives = ['a zebra beside a person', 'a train on a tree']
    values = object_swap_loss(model, tokenizer, prefixes[:2], captions, negatives)
    return {name: float(value.detach().cpu()) for name, value in values.items()}

## Object list and co-occurrence matrix

In [ ]:
object_data = load_object_data(os.path.join(TRACK1_ROOT, 'eval', 'coco_objects.json'))
chair_gt = load_chair_gt(os.path.join(TRACK1_ROOT, 'eval', 'chair_gt_train.pkl'))
categories = object_data['categories']
cooccurrence, category_to_index = build_cooccurrence_matrix(chair_gt, categories)

matrix_path = os.path.join(TRACK3_ROOT, 'eval/cooccurrence_train.pkl')
save_cooccurrence_matrix(matrix_path, cooccurrence, categories)
print('categories:', len(categories))
print('images:', len(chair_gt))
print('matrix:', cooccurrence.shape, cooccurrence.dtype)
print('saved:', matrix_path)
assert np.array_equal(cooccurrence, cooccurrence.T)
assert np.all(np.diag(cooccurrence) >= 0)

categories: 80
images: 113287
matrix: (80, 80) int64
saved: /content/drive/MyDrive/sc4001_nndl_grp_project/phase_1/track_3/eval/cooccurrence_train.pkl


The truth for each image is `seg` union every object list in `cap`; therefore the diagonal counts images containing each object and off-diagonal entries count image-level co-occurrence.

## Swap generation and 100-negative spot checks

In [ ]:
with open(os.path.join(TRACK1_ROOT, 'data', 'karpathy_index.json')) as handle:
    karpathy_index = json.load(handle)

examples = []
for row in karpathy_index:
    if row['split'] != 'train' or row['image_id'] not in chair_gt:
        continue
    image_objects = true_objects(chair_gt[row['image_id']])
    for caption in row['captions']:
        examples.append((caption, image_objects))

structured_check = spot_check_swaps(
    examples, object_data, cooccurrence, category_to_index, n=100, seed=0
)
random_check = spot_check_swaps(
    examples, object_data, cooccurrence, category_to_index,
    n=100, seed=0, random_swap=True,
)
print('structured:', structured_check)
print('random ablation:', random_check)
assert not structured_check['failures']
assert not random_check['failures']
assert structured_check['checked'] > 0
assert random_check['checked'] > 0

structured: {'requested': 100, 'checked': 75, 'failures': []}
random ablation: {'requested': 100, 'checked': 75, 'failures': []}


In [ ]:
example_caption, example_objects = examples[0]
structured = generate_swap(
    example_caption, example_objects, object_data, cooccurrence, category_to_index
)
random_negative = generate_swap(
    example_caption, example_objects, object_data, cooccurrence, category_to_index,
    random_swap=True,
)
print('original:', example_caption)
print('structured:', structured)
print('random:', random_negative)

original: Closeup of bins of food that include broccoli and bread.
structured: {'caption': 'Closeup of bins of food that include baseball bat and bread.', 'source': 'broccoli', 'target': 'baseball bat', 'source_text': 'broccoli', 'random_swap': False}
random: {'caption': 'Closeup of bins of food that include kite and bread.', 'source': 'broccoli', 'target': 'kite', 'source_text': 'broccoli', 'random_swap': True}


## Pretrained ClipCap loss smoke test

Run Track 1's checkpoint-loading cell first. This uses two raw 640-d feature rows; `caption_nll` performs the required L2 normalization internally.

In [ ]:
clipcap_dir = '/content/CLIP_prefix_caption'

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA runtime is required for the pretrained ClipCap smoke test.')
device = 'cuda'

checkpoint_path = os.path.join(
    TRACK1_ROOT, 'models', 'clipcap_transformer_coco_pretrained.pt'
)
if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(
        f'Pretrained checkpoint not found at {checkpoint_path}. Run Track 1 checkpoint setup first.'
    )

PREFIX_LENGTH = 40
ckpt_model = ClipCaptionPrefix(
    PREFIX_LENGTH,
    clip_length=40,
    prefix_size=640,
    num_layers=8,
    mapping_type=MappingType.Transformer,
)
state = torch.load(checkpoint_path, map_location='cpu')
load_result = ckpt_model.load_state_dict(state, strict=False)
assert not load_result.missing_keys, load_result.missing_keys
for key in load_result.unexpected_keys:
    assert key.endswith('.attn.bias') or key.endswith('.attn.masked_bias'), key
ckpt_model = ckpt_model.to(device).eval()
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

feature_path = os.path.join(TRACK1_ROOT, 'features', 'coco_rn50x4_val.pkl')
with open(feature_path, 'rb') as handle:
    val_data = pickle.load(handle)

prefixes = val_data['clip_embedding'][:2].to(device)
loss_values = smoke_test_loss(ckpt_model, tokenizer, prefixes)
print(loss_values)
assert all(np.isfinite(value) for value in loss_values.values())

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


{'positive_nll': 5.884581565856934, 'negative_nll': 6.459476947784424, 'ranking_loss': 0.048545122146606445, 'loss': 5.933126449584961}
